# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
OPENROUTER_API_URL = "https://openrouter.ai/api/v1"
ollama = OpenAI(base_url=OPENROUTER_API_URL, api_key=os.getenv("OPENROUTER_API_KEY"))
MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

In [10]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [11]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'expertise page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'}]}

In [8]:
# Route subsequent requests through OpenRouter
openai = ollama

# Verify the configured client and model
print(f"Using model: {MODEL}")
print(f"Using endpoint: {OPENROUTER_API_URL}")

Using model: nvidia/nemotron-3-ultra-550b-a55b:free
Using endpoint: https://openrouter.ai/api/v1


In [12]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [15]:
edwarddonner=select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling nvidia/nemotron-3-ultra-550b-a55b:free
Found 8 relevant links


In [13]:
huggingface=select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling nvidia/nemotron-3-ultra-550b-a55b:free
Found 15 relevant links


In [14]:
huggingface

{'links': [{'type': 'company profile',
   'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise solutions', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'learning resources', 'url': 'https://huggingface.co/learn'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'join community', 'url': 'https://huggingface.co/join'},
  {'type':

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [16]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling nvidia/nemotron-3-ultra-550b-a55b:free
Found 12 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.8-27B\nUpdated\n2 days ago\n•\n268k\n•\n10k\nmeta-models/Muse-Glimmer-30B\nUpdated\n5 days ago\n•\n293k\n

In [19]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling nvidia/nemotron-3-ultra-550b-a55b:free
Found 15 relevant links


# Hugging Face: The AI Community Building the Future

---

## Company Overview

**Hugging Face** is the world’s leading collaboration platform for the machine learning community. Founded on the belief that AI should be open, ethical, and accessible to all, the company provides a central hub where researchers, engineers, scientists, and enthusiasts share, explore, discover, and experiment with open-source machine learning.

> **Mission:** *Empower the next generation of machine learning engineers, scientists, and end users to learn, collaborate, and share their work to build an open and ethical AI future together.*

With a fast-growing community, the most widely used open-source ML libraries, and a talented science team pushing the boundaries of technology, Hugging Face sits at the heart of the AI revolution.

---

## Platform & Products

| Product | Description | Scale |
|---------|-------------|-------|
| **Models** | Host, version, and collaborate on state-of-the-art models | **2M+** models |
| **Datasets** | Curated, versioned datasets for training & evaluation | **500K+** datasets |
| **Spaces** | Deploy and showcase ML demos & applications | **1M+** apps |
| **Buckets** *(new)* | Scalable storage for large ML artifacts | — |
| **HuggingChat** | Open-source chat interface for LLMs | — |
| **Inference Endpoints** | Dedicated, production-grade model serving | Enterprise |
| **Enterprise Solutions** | Pro, Team, Support, Inference Providers, Private Hubs | — |

**Open-Source Stack:** Transformers, Accelerate, PEFT, TRL, Datasets, Tokenizers, Gradio, and more—used by millions of developers worldwide.

---

## Community & Culture

- **Open by Default:** All public models, datasets, and Spaces are free to use, fork, and extend.
- **Collaboration First:** The Hub is designed for teamwork—discussion threads, pull requests, version history, and organization workspaces.
- **Ethical AI:** Commitment to responsible use, transparency, and bias mitigation; model cards and dataset cards are standard practice.
- **Learning & Sharing:** Daily papers, blog posts, tutorials, courses, and a vibrant Discord/Forum/GitHub ecosystem.
- **Diverse & Global:** Contributors from academia, industry, and hobbyist communities across every continent.

**Community Stats:**
- **103K+** followers on the Hub
- Thousands of active contributors daily
- Regular research publications from the Hugging Face science team

---

## Customers & Use Cases

| Segment | Examples |
|---------|----------|
| **Enterprises** | NVIDIA, Microsoft, AWS, Google, Meta, Salesforce, Bloomberg, Pfizer, and thousands more |
| **Startups** | Building MVPs on Spaces, fine-tuning models via AutoTrain, deploying via Inference Endpoints |
| **Researchers** | Sharing benchmarks, reproducing results, distributing artifacts (e.g., FineWeb, The Stack, HH-RLHF) |
| **Educators & Students** | Free GPU access, course materials, hands-on demos |
| **Open-Source Projects** | Hosting canonical model weights (Llama, Qwen, Mistral, Stable Diffusion, Whisper, etc.) |

---

## Careers

Hugging Face is a **remote-first, globally distributed team** hiring across engineering, research, developer relations, product, design, sales, and operations.

- **Culture:** High autonomy, strong writing culture, open-source contribution expected, quarterly on-sites.
- **Benefits:** Competitive salary + equity, comprehensive health, learning budget, generous PTO, hardware allowance.
- **Open Roles:** Check the [Careers page](https://huggingface.co/careers) for current openings—positions range from ML Research Engineers to Community Managers to Enterprise Account Executives.

---

## Key Links

- **Hub:** [huggingface.co](https://huggingface.co)
- **Documentation:** [huggingface.co/docs](https://huggingface.co/docs)
- **Blog:** [huggingface.co/blog](https://huggingface.co/blog)
- **Forum & Discord:** Community support & discussion
- **GitHub:** [github.com/huggingface](https://github.com/huggingface)
- **Enterprise:** [huggingface.co/enterprise](https://huggingface.co/enterprise)
- **Careers:** [huggingface.co/careers](https://huggingface.co/careers)
- **Brand Assets:** [huggingface.co/brand-assets](https://huggingface.co/brand-assets)

---

*Hugging Face — where the machine learning community builds the future, together.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [21]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling nvidia/nemotron-3-ultra-550b-a55b:free
Found 21 relevant links


# Hugging Face: The AI Community Building the Future

---

## Company Overview

**Hugging Face** is the world's leading collaboration platform for machine learning, serving as the central hub where the global AI community creates, discovers, and collaborates on models, datasets, and applications. With a mission to democratize good machine learning, the company has become the **home of machine learning** for researchers, developers, and enterprises alike.

> **"The platform where the machine learning community collaborates on models, datasets, and applications."**

---

## Platform Scale & Reach

| Resource | Volume | Description |
|----------|--------|-------------|
| **Models** | 2M+ | Pre-trained models for NLP, vision, audio, and multimodal tasks |
| **Datasets** | 500k+ | Curated datasets for training and evaluation |
| **Spaces (Apps)** | 1M+ | Interactive ML demos and applications |
| **Community** | 100k+ followers | Active contributors and organizations |

---

## Core Offerings

### 🤗 Open Collaboration Platform
- **Unlimited public hosting** for models, datasets, and applications
- **Version control & collaboration** built for ML workflows
- **Discoverability** through trending, tasks, languages, and collections

### 🏢 Enterprise Solutions
- **Hugging Face PRO** — Enhanced features for power users
- **Enterprise Support** — Dedicated SLAs and expertise
- **Inference Endpoints** — Secure, scalable model deployment
- **Inference Providers** — Partner ecosystem for compute
- **Storage Buckets** — New enterprise-grade data storage

### 🛠️ Open Source Ecosystem
- **Transformers** — State-of-the-art ML library (100k+ GitHub stars)
- **Datasets / Tokenizers / Accelerate / PEFT** — Core tooling stack
- **Hardware partnerships** — Optimized for leading accelerators

---

## Community & Culture

### Open-First Philosophy
Hugging Face operates on **radical openness** — the platform is built by and for the community. Public resources are free forever, fostering a culture of sharing, reproducibility, and collective advancement.

### Vibrant Knowledge Sharing
- **Daily Papers** — Latest research distilled
- **Blog & Posts** — Technical deep-dives and tutorials
- **Learn** — Structured courses and certifications
- **Discord & Forum** — Real-time peer support
- **HuggingChat** — Open-source chat interface

### Diverse & Global
With contributors from **academia, industry, and independent researchers** worldwide, the community spans 100+ languages and every ML domain — from LLMs and diffusion models to specialized scientific applications.

---

## Who Uses Hugging Face

| Segment | Use Cases |
|---------|-----------|
| **Researchers** | Publish models, benchmark on standard datasets, reproduce results |
| **ML Engineers** | Prototype in Spaces, deploy via Inference Endpoints, fine-tune with PEFT |
| **Enterprises** | Private model hubs, compliance-ready deployment, dedicated support |
| **Startups** | Accelerate time-to-market with pre-trained models and demos |
| **Educators & Students** | Free access to SOTA models, hands-on learning environments |

**Notable organizations** hosting on the hub include Meta, Microsoft, Google, NVIDIA, Stability AI, and thousands of universities and startups.

---

## Technology Highlights

- **Model Hub** — Git-based versioning, model cards, license tags, inference API
- **Dataset Hub** — Streaming, splitting, formatting, SQL queries
- **Spaces** — Zero-config Gradio/Streamlit/Docker apps with free GPU (ZeroGPU)
- **AutoTrain** — No-code fine-tuning and deployment
- **Optimum** — Hardware-specific optimization (ONNX, TensorRT, OpenVINO)

---

## Careers & Talent

Hugging Face attracts top talent passionate about **open science, developer experience, and ML infrastructure**. While specific roles aren't listed here, the company typically hires for:

- **Engineering** — Platform, inference, open source libraries
- **Research** — Model architecture, alignment, efficiency
- **Developer Relations** — Community building, content, events
- **Sales & Solutions** — Enterprise adoption, technical consulting
- **Product & Design** — UX for ML workflows, hub features

**Culture signals:** Remote-friendly, open-source contributors valued, conference presence (NeurIPS, ICML, ICLR), active blog authorship.

---

## Get Started

| Action | Link |
|--------|------|
| Explore Models | `huggingface.co/models` |
| Browse Datasets | `huggingface.co/datasets` |
| Try Spaces | `huggingface.co/spaces` |
| Read Docs | `huggingface.co/docs` |
| Enterprise Inquiry | `huggingface.co/enterprise` |
| Join Community | Discord / Forum / GitHub |

---

*Building the future of AI, together.* 🤗

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>